In [1]:
import pandas as pd   
import numpy as np  
import polars as pl

In [2]:
df_search_queries = pd.read_csv('../Data/002/search_queries.csv',parse_dates=['query_date'])
df_users = pd.read_csv('../Data/002/users.csv',parse_dates=['signup_date'])

In [3]:
pl_search_queries = pl.read_csv('../Data/002/search_queries.csv', try_parse_dates=True)
pl_users = pl.read_csv('../Data/002/users.csv', try_parse_dates=True)

# Pregunta 1

### ¿Cuántas consultas de búsqueda (search queries) tuvieron un clic en un enlace o un tiempo de permanencia (dwell time) de más de 30 segundos en octubre de 2024?

```SQL
SELECT
    COUNT(*)
FROM search_queries
WHERE (clicks > 0 OR dwell_time_seconds > 30) AND
      query_date BETWEEN '2024-10-01' AND '2024-10-31'
```

In [10]:
oct = df_search_queries[
    (df_search_queries['query_date'].between('2024-10-01','2024-10-31')) &
    ((df_search_queries['clicks'] > 0) | (df_search_queries['dwell_time_seconds'] > 30))
].shape[0]

oct

15

In [13]:
from datetime import date

res = pl_search_queries.filter(
    (pl.col('query_date').is_between(date(2024,10,1),date(2024,10,31))) &
    ((pl.col('clicks') > 0) | (pl.col('dwell_time_seconds') > 30))
).height

res

15

# Pregunta 2

### ¿Puedes averiguar cuántas consultas de búsqueda en octubre de 2024 fueron realizadas por usuarios que hicieron clic en un enlace Y pasaron más de 30 segundos en la página de resultados de búsqueda?

```SQL
SELECT
    COUNT(*)
FROM search_queries
WHERE (query_date BETWEEN '2024-10-01' AND '2024-10-31') AND
      ((clicks > 0) AND (dwell_time_seconds > 30))
```

In [6]:
oct = df_search_queries[
    (df_search_queries['query_date'].between('2024-10-01','2024-10-31')) &
    ((df_search_queries['clicks'] > 0) & (df_search_queries['dwell_time_seconds'] > 30))
].shape[0]

In [9]:
from datetime import date

oct = pl_search_queries.filter(
    (pl.col('query_date').is_between(date(2024,10,1),date(2024,10,31))) &
    ((pl.col('clicks') > 0) & (pl.col('dwell_time_seconds') > 30))
).height

oct

8

# Pregunta 3

### Para los usuarios que se registraron en la primera semana de octubre de 2024 (por ejemplo, del 1 al 7 de octubre), ¿cuántas consultas de búsqueda realizaron en total?

```SQL
SELECT
    COUNT(s.query_id)
FROM search_queries AS s
INNER JOIN users AS u ON s.user_id = u.user_id
WHERE u.signup_date BETWEEN '2024-10-01' AND '2024-10-07';
```

In [10]:
oct_merge = df_search_queries.merge(
    df_users,
    on = 'user_id', 
    how = 'inner'
)

oct_w1 = oct_merge[
    oct_merge['signup_date'].between('2024-10-01','2024-10-07')
].shape[0]

oct_w1

2

In [11]:
oct_join = pl_search_queries.join(
    pl_users,
    on = 'user_id',
    how = 'inner'
)

oct_w1 = oct_join.filter(
    pl.col('signup_date').is_between(date(2024,10,1),date(2024,10,7))
).height

oct_w1

2